# Assignment Beta, Question 4: LLM Evaluation
**Student ID:** 23k-0078

Sentiment classification of 300 reviews, plus evaluation of one generated justification.

In [1]:
%pip install -q nltk rouge-score
import math
from collections import Counter

import nltk
import numpy as np
import pandas as pd

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
pd.set_option("display.precision", 4)

classes = ["Positive", "Negative", "Neutral"]
stage1 = np.array([[100, 5, 15], [8, 80, 12], [20, 10, 50]])
stage2 = np.array([[106, 4, 10], [6, 84, 10], [9, 7, 64]])

def evaluate(cm, z=1.96):
    n = cm.sum()
    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * precision * recall / (precision + recall)
    support = cm.sum(axis=1)
    acc = tp.sum() / n
    se = math.sqrt(acc * (1 - acc) / n)
    per_class = pd.DataFrame({"TP": tp, "FP": fp, "FN": fn, "Support": support,
                              "Precision": precision, "Recall": recall, "F1": f1}, index=classes)
    return {"n": n, "correct": tp.sum(), "accuracy": acc, "se": se,
            "ci": (acc - z * se, acc + z * se), "per_class": per_class,
            "macro_f1": f1.mean(), "weighted_f1": (f1 * support).sum() / n,
            "micro_f1": 2 * tp.sum() / (2 * tp.sum() + fp.sum() + fn.sum())}

def report(name, r):
    print(f"--- {name} ---")
    print(f"Accuracy = {r['correct']}/{r['n']} = {r['accuracy']:.4f}, SE = {r['se']:.4f}")
    print(f"95% CI = {r['accuracy']:.4f} +/- {1.96 * r['se']:.4f} = [{r['ci'][0]:.4f}, {r['ci'][1]:.4f}]")
    print(r["per_class"].round(4).to_string())
    print(f"Macro F1 = {r['macro_f1']:.4f}, Weighted F1 = {r['weighted_f1']:.4f}, Micro F1 = {r['micro_f1']:.4f}")

r1, r2 = evaluate(stage1), evaluate(stage2)


[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: C:\Users\arsha\AppData\Local\Temp\claude\C--Users-arsha-OneDrive-Desktop-agentic-ai-assignment-2\681da536-46e9-4a00-bde4-507f769b8570\scratchpad\venv\Scripts\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Parts 1 to 3: Stage 1 accuracy, confidence interval, per-class metrics and averaged F1

In [2]:
report("Stage 1 (zero-shot)", r1)

--- Stage 1 (zero-shot) ---
Accuracy = 230/300 = 0.7667, SE = 0.0244
95% CI = 0.7667 +/- 0.0479 = [0.7188, 0.8145]
           TP  FP  FN  Support  Precision  Recall      F1
Positive  100  28  20      120     0.7812  0.8333  0.8065
Negative   80  15  20      100     0.8421  0.8000  0.8205
Neutral    50  27  30       80     0.6494  0.6250  0.6369
Macro F1 = 0.7546, Weighted F1 = 0.7659, Micro F1 = 0.7667


## Part 4: Stage 1 error analysis
Each off-diagonal cell, with the share of that actual class it represents.

In [3]:
errors = []
for i, a in enumerate(classes):
    for j, p in enumerate(classes):
        if i != j:
            errors.append((a, p, stage1[i, j], stage1[i, j] / stage1[i].sum()))
err = pd.DataFrame(errors, columns=["Actual", "Predicted", "Count", "Share of actual"]).sort_values("Count", ascending=False)
print(err.round(4).to_string(index=False))
total_errors = stage1.sum() - np.trace(stage1)
involving_neutral = err[(err.Actual == "Neutral") | (err.Predicted == "Neutral")].Count.sum()
print(f"Total errors = {total_errors}; errors involving Neutral = {involving_neutral} ({involving_neutral / total_errors:.1%})")

  Actual Predicted  Count  Share of actual
 Neutral  Positive     20           0.2500
Positive   Neutral     15           0.1250
Negative   Neutral     12           0.1200
 Neutral  Negative     10           0.1250
Negative  Positive      8           0.0800
Positive  Negative      5           0.0417
Total errors = 70; errors involving Neutral = 57 (81.4%)


**Interpretation.** Neutral is the weak class (F1 0.637, against 0.807 and 0.821), and 57 of the 70 errors involve it. The largest single error is Neutral predicted as Positive (20 reviews, 25% of Neutral). Direct Positive and Negative confusions account for only 13 errors, so the model separates polarity well and fails at the boundary of Neutral.

**Recommendations.** Define Neutral explicitly in the prompt, add few-shot examples that include mixed reviews, read a sample of the Neutral errors to confirm their cause, and check whether human annotators agree on Neutral labels. Track per-class F1 and macro F1, because overall accuracy hides this weakness.

## Part 5: Stage 2 metrics and comparison with Stage 1

In [4]:
report("Stage 2 (definition + few-shot)", r2)
rows = {"Accuracy": (r1["accuracy"], r2["accuracy"]), "Macro F1": (r1["macro_f1"], r2["macro_f1"]),
        "Weighted F1": (r1["weighted_f1"], r2["weighted_f1"]), "Micro F1": (r1["micro_f1"], r2["micro_f1"])}
for c in classes:
    rows[f"{c} precision"] = (r1["per_class"].loc[c, "Precision"], r2["per_class"].loc[c, "Precision"])
    rows[f"{c} recall"] = (r1["per_class"].loc[c, "Recall"], r2["per_class"].loc[c, "Recall"])
    rows[f"{c} F1"] = (r1["per_class"].loc[c, "F1"], r2["per_class"].loc[c, "F1"])
cmp = pd.DataFrame(rows, index=["Stage 1", "Stage 2"]).T
cmp["Change"] = cmp["Stage 2"] - cmp["Stage 1"]
print(cmp.round(4).to_string())
print("Neutral-to-Positive errors:", stage1[2, 0], "->", stage2[2, 0])
print("Stage 1 CI:", np.round(r1["ci"], 4), " Stage 2 CI:", np.round(r2["ci"], 4))
e1, e2 = stage1.sum() - np.trace(stage1), stage2.sum() - np.trace(stage2)
print(f"Errors: {e1} -> {e2} ({(e2 - e1) / e1:+.1%})")
neutral_errors_2 = stage2[2].sum() - stage2[2, 2] + stage2[:, 2].sum() - stage2[2, 2]
print(f"Stage 2 errors involving Neutral = {neutral_errors_2} ({neutral_errors_2 / e2:.1%})")

--- Stage 2 (definition + few-shot) ---
Accuracy = 254/300 = 0.8467, SE = 0.0208
95% CI = 0.8467 +/- 0.0408 = [0.8059, 0.8874]
           TP  FP  FN  Support  Precision  Recall      F1
Positive  106  15  14      120     0.8760  0.8833  0.8797
Negative   84  11  16      100     0.8842  0.8400  0.8615
Neutral    64  20  16       80     0.7619  0.8000  0.7805
Macro F1 = 0.8406, Weighted F1 = 0.8472, Micro F1 = 0.8467
                    Stage 1  Stage 2  Change
Accuracy             0.7667   0.8467  0.0800
Macro F1             0.7546   0.8406  0.0859
Weighted F1          0.7659   0.8472  0.0812
Micro F1             0.7667   0.8467  0.0800
Positive precision   0.7812   0.8760  0.0948
Positive recall      0.8333   0.8833  0.0500
Positive F1          0.8065   0.8797  0.0732
Negative precision   0.8421   0.8842  0.0421
Negative recall      0.8000   0.8400  0.0400
Negative F1          0.8205   0.8615  0.0410
Neutral precision    0.6494   0.7619  0.1126
Neutral recall       0.6250   0.8000  0.17

**Comparison.** Every class improves, and Neutral improves most (F1 +0.1435, recall 0.625 to 0.800). Macro F1 gains slightly more than weighted F1 because the smallest and weakest class gained most. Neutral is still the weakest class and is still involved in most remaining errors.

**Caveats.** The two 95% intervals overlap slightly. That comparison treats the stages as independent even though both use the same reviews; a paired McNemar test would be stronger, but it needs per-review results that are not given. The prompt was also revised from errors on these same 300 reviews, so the gain may be optimistic and should be confirmed on a held-out set.

## Part 6: Prompt stability across five example orderings

In [5]:
runs = np.array([0.847, 0.840, 0.853, 0.837, 0.850])
print(f"Mean = {runs.mean():.4f}")
print("Squared deviations:", np.round((runs - runs.mean())**2, 8))
print(f"Sample SD (n-1) = {runs.std(ddof=1):.4f}; population SD (n) = {runs.std(ddof=0):.4f}")
print(f"Range = {runs.max() - runs.min():.4f}; in reviews out of 300 = {(runs.max() - runs.min()) * 300:.1f}")
print(f"Coefficient of variation = {runs.std(ddof=1) / runs.mean():.2%}")
print(f"Sampling SE from Stage 2 = {r2['se']:.4f}")
print(f"Stage gain / SD = {(r2['accuracy'] - r1['accuracy']) / runs.std(ddof=1):.1f}")

Mean = 0.8454
Squared deviations: [2.560e-06 2.916e-05 5.776e-05 7.056e-05 2.116e-05]
Sample SD (n-1) = 0.0067; population SD (n) = 0.0060
Range = 0.0160; in reviews out of 300 = 4.8
Coefficient of variation = 0.80%
Sampling SE from Stage 2 = 0.0208
Stage gain / SD = 11.9


**Comment.** Accuracy is 0.8454 ± 0.0067 (sample SD, since the five runs are a sample of the 720 possible orderings of six examples). The prompt is stable at the aggregate level: the full range is about 5 reviews out of 300, the SD is about a third of the sampling error, and the Stage 1 to Stage 2 gain is about 12 times the SD. Five runs is a small sample, and equal accuracy does not prove the same reviews were classified the same way, so per-review agreement should also be checked.

## Part 7: BLEU-4, ROUGE-1, ROUGE-2, ROUGE-L and METEOR
Each metric is computed from its definition and cross-checked against a standard library. No stemming is applied in ROUGE.

In [6]:
reference = "the delivery was late but the product quality is good".split()
candidate = "the product quality is good but delivery was slow".split()
print(f"Reference length r = {len(reference)}, candidate length c = {len(candidate)}")

def ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))

# BLEU-4 with clipped counts, uniform weights and no smoothing.
precisions = []
for n in range(1, 5):
    c_grams, r_grams = ngrams(candidate, n), ngrams(reference, n)
    matched = sum(min(count, r_grams[g]) for g, count in c_grams.items())
    total = sum(c_grams.values())
    precisions.append(matched / total)
    print(f"{n}-gram precision = {matched}/{total} = {matched / total:.4f}; matches: {[' '.join(g) for g in c_grams if g in r_grams]}")
bp = math.exp(1 - len(reference) / len(candidate)) if len(candidate) < len(reference) else 1.0
geo = math.exp(sum(math.log(p) for p in precisions) / 4)
bleu = bp * geo
print(f"Brevity penalty = exp(1 - {len(reference)}/{len(candidate)}) = {bp:.4f}")
print(f"Geometric mean = {geo:.4f}; BLEU-4 = {bleu:.4f}")

from nltk.translate.bleu_score import sentence_bleu
print(f"NLTK check: {sentence_bleu([reference], candidate):.4f}")

Reference length r = 10, candidate length c = 9
1-gram precision = 8/9 = 0.8889; matches: ['the', 'product', 'quality', 'is', 'good', 'but', 'delivery', 'was']
2-gram precision = 5/8 = 0.6250; matches: ['the product', 'product quality', 'quality is', 'is good', 'delivery was']
3-gram precision = 3/7 = 0.4286; matches: ['the product quality', 'product quality is', 'quality is good']
4-gram precision = 2/6 = 0.3333; matches: ['the product quality is', 'product quality is good']
Brevity penalty = exp(1 - 10/9) = 0.8948
Geometric mean = 0.5308; BLEU-4 = 0.4750
NLTK check: 0.4750


In [7]:
def prf(match, c_total, r_total):
    p, r = match / c_total, match / r_total
    return p, r, 2 * p * r / (p + r)

def rouge_n(n):
    c_grams, r_grams = ngrams(candidate, n), ngrams(reference, n)
    return sum(min(v, r_grams[g]) for g, v in c_grams.items()), sum(c_grams.values()), sum(r_grams.values())

def lcs_table(a, b):
    t = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i, x in enumerate(a, 1):
        for j, w in enumerate(b, 1):
            t[i][j] = t[i - 1][j - 1] + 1 if x == w else max(t[i - 1][j], t[i][j - 1])
    return t

def lcs_tokens(a, b):
    t, out, i, j = lcs_table(a, b), [], len(a), len(b)
    while i and j:
        if a[i - 1] == b[j - 1]:
            out.append(a[i - 1]); i -= 1; j -= 1
        elif t[i - 1][j] >= t[i][j - 1]:
            i -= 1
        else:
            j -= 1
    return out[::-1]

results = {}
for name, n in [("ROUGE-1", 1), ("ROUGE-2", 2)]:
    m, ct, rt = rouge_n(n)
    results[name.lower().replace("-", "")] = prf(m, ct, rt)
    print(f"{name}: matches = {m}, candidate = {ct}, reference = {rt}: P/R/F = {np.round(prf(m, ct, rt), 4)}")
lcs = lcs_tokens(candidate, reference)
results["rougel"] = prf(len(lcs), len(candidate), len(reference))
print(f"ROUGE-L: LCS = {lcs!r}, length {len(lcs)}: P/R/F = {np.round(results['rougel'], 4)}")

from rouge_score import rouge_scorer
lib = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False).score(" ".join(reference), " ".join(candidate))
for k, v in lib.items():
    print(f"Library {k}: P={v.precision:.4f} R={v.recall:.4f} F={v.fmeasure:.4f}")

ROUGE-1: matches = 8, candidate = 9, reference = 10: P/R/F = [0.8889 0.8    0.8421]
ROUGE-2: matches = 5, candidate = 8, reference = 9: P/R/F = [0.625  0.5556 0.5882]
ROUGE-L: LCS = ['the', 'product', 'quality', 'is', 'good'], length 5: P/R/F = [0.5556 0.5    0.5263]
Library rouge1: P=0.8889 R=0.8000 F=0.8421
Library rouge2: P=0.6250 R=0.5556 F=0.5882
Library rougeL: P=0.5556 R=0.5000 F=0.5263


In [8]:
# METEOR, Banerjee and Lavie (2005): Fmean = 10PR / (R + 9P), penalty = 0.5 * (chunks / matches)^3.
# Exact alignment that gives the fewest chunks, with reference positions counted from 1:
alignment = {"the": 6, "product": 7, "quality": 8, "is": 9, "good": 10, "but": 5, "delivery": 2, "was": 3}
positions = list(alignment.values())
chunks = 1 + sum(1 for a, b in zip(positions, positions[1:]) if b != a + 1)

def meteor(m, chunks):
    p, r = m / len(candidate), m / len(reference)
    fmean = 10 * p * r / (r + 9 * p)
    penalty = 0.5 * (chunks / m) ** 3
    return p, r, fmean, penalty, fmean * (1 - penalty)

m = len(positions)
p, r, fmean, penalty, score = meteor(m, chunks)
print(f"Matches = {m}, chunks = {chunks}, unmatched: candidate 'slow', reference 'late' and the first 'the'")
print(f"P = {p:.4f}, R = {r:.4f}, Fmean = {fmean:.4f}, penalty = {penalty:.4f}, METEOR = {score:.4f}")

from nltk.corpus import wordnet as wn
from nltk.translate.meteor_score import single_meteor_score
print(f"NLTK check (exact, stem and WordNet stages): {single_meteor_score(reference, candidate):.4f}")
slow, late = set(), set()
for s in wn.synsets("slow"):
    slow.update(s.lemma_names())
for s in wn.synsets("late"):
    late.update(s.lemma_names())
print("WordNet lemma of 'late' listed for 'slow':", "late" in slow, "; 'slow' listed for 'late':", "slow" in late)
print("Shared lemmas:", sorted(slow & late))

# Sensitivity: if slow/late were accepted as a synonym match, 'delivery was slow' joins one chunk.
print("If 'slow' matched 'late': P, R, Fmean, penalty, METEOR =", np.round(meteor(9, 3), 4))

Matches = 8, chunks = 3, unmatched: candidate 'slow', reference 'late' and the first 'the'
P = 0.8889, R = 0.8000, Fmean = 0.8081, penalty = 0.0264, METEOR = 0.7868


NLTK check (exact, stem and WordNet stages): 0.7868
WordNet lemma of 'late' listed for 'slow': False ; 'slow' listed for 'late': False
Shared lemmas: ['tardily']
If 'slow' matched 'late': P, R, Fmean, penalty, METEOR = [1.     0.9    0.9091 0.0185 0.8923]


**Interpretation.** Word overlap is high (ROUGE-1 F1 0.842), but the scores fall as word order starts to matter (ROUGE-2 0.588, ROUGE-L 0.526, BLEU-4 0.475) because the candidate swaps the two clauses. METEOR stays high (0.787) because it scores aligned words and only lightly penalises the three chunks.

None of these metrics credits *slow* as a paraphrase of *late*, since WordNet does not list them as synonyms here. They also miss a real change of emphasis: "X but Y" stresses Y, so the reference stresses good quality while the candidate stresses poor delivery.

## Part 8: Perplexity of the generated justification
$PP = \exp\left(-\frac{1}{N}\sum_i \ln P_i\right)$

In [9]:
probs = np.array([0.60, 0.40, 0.50, 0.80, 0.70, 0.30, 0.25, 0.50, 0.20])
for tok, pr in zip(candidate, probs):
    print(f"{tok:9s} P = {pr:.2f}, ln P = {math.log(pr):+.4f}")
sum_log = np.log(probs).sum()
avg_nll = -sum_log / len(probs)
print(f"Sum ln P = {sum_log:.4f}; average NLL = {avg_nll:.4f} nats; perplexity = exp({avg_nll:.4f}) = {math.exp(avg_nll):.4f}")
print(f"Geometric mean token probability = {1 / math.exp(avg_nll):.4f}")

the       P = 0.60, ln P = -0.5108
product   P = 0.40, ln P = -0.9163
quality   P = 0.50, ln P = -0.6931
is        P = 0.80, ln P = -0.2231
good      P = 0.70, ln P = -0.3567
but       P = 0.30, ln P = -1.2040
delivery  P = 0.25, ln P = -1.3863
was       P = 0.50, ln P = -0.6931
slow      P = 0.20, ln P = -1.6094
Sum ln P = -7.5929; average NLL = 0.8437 nats; perplexity = exp(0.8437) = 2.3249
Geometric mean token probability = 0.4301


**Interpretation.** A perplexity of 2.32 means the model was, on average, as uncertain as choosing uniformly between about 2.3 tokens at each step. That is moderate confidence. Its least confident tokens are *but*, *delivery* and *slow* (0.30, 0.25 and 0.20), which is where the sentence changes clause and where it departs from the reference.

Perplexity measures the model's confidence in its own output, not correctness, so a model can be confidently wrong. A single nine-token sentence is also a very small sample, and values are only comparable between models that share a tokenizer.